In [4]:
import tkinter as tk
from tkinter import messagebox, ttk
import pandas as pd
import os

# Book Class
class Book:
    def __init__(self, ref_number, title, author):
        self.ref_number = ref_number
        self.title = title
        self.author = author
        self.is_trending = False
        self.left = None
        self.right = None

# Book Binary Search Tree Class
class BookBST:
    def __init__(self):
        self.root = None

    def insert(self, book):
        if not self.root:
            self.root = book
        else:
            self._insert_rec(self.root, book)

    def _insert_rec(self, current, book):
        if book.ref_number < current.ref_number:
            if current.left is None:
                current.left = book
            else:
                self._insert_rec(current.left, book)
        else:
            if current.right is None:
                current.right = book
            else:
                self._insert_rec(current.right, book)

    def delete(self, ref_number):
        self.root = self._delete_rec(self.root, ref_number)

    def _delete_rec(self, current, ref_number):
        if not current:
            return current
        if ref_number < current.ref_number:
            current.left = self._delete_rec(current.left, ref_number)
        elif ref_number > current.ref_number:
            current.right = self._delete_rec(current.right, ref_number)
        else:
            if current.left is None:
                return current.right
            elif current.right is None:
                return current.left
            temp = self._min_value_node(current.right)
            current.ref_number, current.title, current.author = temp.ref_number, temp.title, temp.author
            current.right = self._delete_rec(current.right, temp.ref_number)
        return current

    def _min_value_node(self, node):
        current = node
        while current.left is not None:
            current = current.left
        return current

    def inorder_traversal(self):
        books = []
        self._inorder_traversal(self.root, books)
        return books

    def _inorder_traversal(self, current, books):
        if current:
            self._inorder_traversal(current.left, books)
            books.append(current)
            self._inorder_traversal(current.right, books)

    def search(self, ref_number):
        return self._search_rec(self.root, ref_number)

    def _search_rec(self, current, ref_number):
        if current is None or current.ref_number == ref_number:
            return current
        if ref_number < current.ref_number:
            return self._search_rec(current.left, ref_number)
        return self._search_rec(current.right, ref_number)

# Initialize main data structures
bst = BookBST()
excel_path = "Book1.xlsx"

def load_books_from_excel():
    if os.path.exists(excel_path):
        df = pd.read_excel(excel_path)
        for _, row in df.iterrows():
            book = Book(int(row['Reference Number']), row['Title'], row['Author'])
            book.is_trending = row.get('Trending', 'No') == "Yes"
            bst.insert(book)

def save_books_to_excel():
    books = bst.inorder_traversal()
    data = {
        'Reference Number': [book.ref_number for book in books],
        'Title': [book.title for book in books],
        'Author': [book.author for book in books],
        'Trending': ["Yes" if book.is_trending else "No" for book in books]
    }
    df = pd.DataFrame(data)
    df.to_excel(excel_path, index=False)

class BookstoreApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bookstore Management System")
        self.geometry("900x600")

        # Load books from Excel on initialization
        load_books_from_excel()

        # Create tabs
        self.tab_control = ttk.Notebook(self)
        
        # Create Admin Tab
        self.admin_tab = AdminTab(self.tab_control)
        self.tab_control.add(self.admin_tab, text='Admin')
        
        # Create User Tab
        self.user_tab = UserTab(self.tab_control)
        self.tab_control.add(self.user_tab, text='User ')
        
        self.tab_control.pack(expand=1, fill='both')

class AdminTab(ttk.Frame):
    def __init__(self, parent):
        super().__init__(parent)
        self.search_history = []  # Initialize search history for admin
        self.setup_ui()

    def setup_ui(self):
        # Book addition form
        ttk.Label(self, text="Add New Book").grid(column=0, row=0, padx= 10, pady=10)

        self.ref_input = ttk.Entry(self)
        self.ref_input.insert(0, "Reference Number")
        self.ref_input.grid(column=0, row=1, padx=10, pady=5)

        self.title_input = ttk.Entry(self)
        self.title_input.insert(0, "Title")
        self.title_input.grid(column=0, row=2, padx=10, pady=5)

        self.author_input = ttk.Entry(self)
        self.author_input.insert(0, "Author")
        self.author_input.grid(column=0, row=3, padx=10, pady=5)

        add_book_button = ttk.Button(self, text="Add Book", command=self.add_book)
        add_book_button.grid(column=0, row=4, padx=10, pady=10)

        delete_book_button = ttk.Button(self, text="Delete Book", command=self.delete_book)
        delete_book_button.grid(column=1, row=4, padx=10, pady=10)

        mark_trending_button = ttk.Button(self, text="Mark as Trending", command=self.mark_as_trending)
        mark_trending_button.grid(column=2, row=4, padx=10, pady=10)

        unmark_trending_button = ttk.Button(self, text="Unmark as Trending", command=self.unmark_as_trending)
        unmark_trending_button.grid(column=3, row=4, padx=10, pady=10)

        # Book view table
        self.book_table = ttk.Treeview(self, columns=("Ref Number", "Title", "Author", "Trending"), show='headings')
        self.book_table.heading("Ref Number", text="Reference Number")
        self.book_table.heading("Title", text="Title")
        self.book_table.heading("Author", text="Author")
        self.book_table.heading("Trending", text="Trending")
        self.book_table.grid(column=0, row=5, columnspan=4, padx=10, pady=10)

        # Trending books table
        ttk.Label(self, text="Trending Books").grid(column=1, row=5, padx=10, pady=10)
        self.trending_table = TrendingTable(self)
        self.trending_table.grid(column=1, row=6, padx=10, pady=10)

        # Search functionality for admin
        ttk.Label(self, text="Search Book by Reference Number").grid(column=0, row=8, padx=10, pady=10)
        self.admin_search_input = ttk.Entry(self)
        self.admin_search_input.grid(column=0, row=9, padx=10, pady=5)
        
        search_button = ttk.Button(self, text="Search", command=self.admin_search_book)
        search_button.grid(column=1, row=9, padx=10, pady=5)

        # Admin search history display
        ttk.Label(self, text="Admin Search History").grid(column=0, row=10, padx=10, pady=10)
        self.admin_search_history_box = tk.Listbox(self, height=5, width=50)
        self.admin_search_history_box.grid(column=0, row=11, columnspan=2, padx=10, pady=10)

        self.load_books()

    def add_book(self):
        ref_number = self.ref_input.get()
        title = self.title_input.get()
        author = self.author_input.get()

        if not ref_number.isdigit():
            messagebox.showwarning("Input Error", "Please enter a valid reference number.")
            return
        
        book = Book(int(ref_number), title, author)
        bst.insert(book)
        self.load_books()
        self.ref_input.delete(0, tk.END)
        self.title_input.delete(0, tk.END)
        self.author_input.delete(0, tk.END)
        messagebox.showinfo("Success", "Book added successfully.")

    def delete_book(self):
        selected_item = self.book_table.selection()
        if not selected_item:
            messagebox.showwarning("Selection Error", "Please select a book to delete.")
            return

        ref_number = self.book_table.item(selected_item)['values'][0]
        bst.delete(int(ref_number))
        self.load_books()
        messagebox.showinfo("Success", "Book deleted successfully.")

    def mark_as_trending(self):
        selected_item = self.book_table.selection()
        if not selected_item:
            messagebox.showwarning("Selection Error", "Please select a book to mark as trending.")
            return

        ref_number = self.book_table.item(selected_item)['values'][0]
        book = bst.search(int(ref_number))
        if book:
            book.is_trending = True
            self.load_books()
            self.trending_table.load_trending_books()
            messagebox.showinfo("Success", "Book marked as trending.")
        else:
            messagebox.showwarning("Error", "Book not found.")

    def unmark_as_trending(self):
        selected_item = self .book_table.selection()
        if not selected_item:
            messagebox.showwarning("Selection Error", "Please select a book to unmark as trending.")
            return

        ref_number = self.book_table.item(selected_item)['values'][0]
        book = bst.search(int(ref_number))
        if book:
            book.is_trending = False
            self.load_books()
            self.trending_table.load_trending_books()
            messagebox.showinfo("Success", "Book unmarked as trending.")
        else:
            messagebox.showwarning("Error", "Book not found.")

    def admin_search_book(self):
        ref_number = self.admin_search_input.get()
        if not ref_number.isdigit():
            messagebox.showwarning("Input Error", "Please enter a valid reference number.")
            return

        book = bst.search(int(ref_number))
        if book:
            messagebox.showinfo("Search Result", f"Book Found:\nReference Number: {book.ref_number}\nTitle: {book.title}\nAuthor: {book.author}")
            self.admin_search_history_box.insert(tk.END, f"{book.ref_number} - {book.title} by {book.author}")
            self.search_history.append(book.ref_number)
        else:
            messagebox.showinfo("Search Result", "Book not found.")
    
    def load_books(self):
        for row in self.book_table.get_children():
            self.book_table.delete(row)
        for book in bst.inorder_traversal():
            self.book_table.insert("", "end", values=(book.ref_number, book.title, book.author, "Yes" if book.is_trending else "No"))

class UserTab(ttk.Frame):
    def __init__(self, parent):
        super().__init__(parent)
        self.search_history = []  # Initialize search history for user
        self.setup_ui()

    def setup_ui(self):
        # User search interface
        ttk.Label(self, text="Search Book by Reference Number").grid(column=0, row=0, padx=10, pady=10)
        self.user_search_input = ttk.Entry(self)
        self.user_search_input.grid(column=0, row=1, padx=10, pady=5)

        search_button = ttk.Button(self, text="Search", command=self.user_search_book)
        search_button.grid(column=1, row=1, padx=10, pady=5)

        # User search history display
        ttk.Label(self, text="User Search History").grid(column=0, row=2, padx=10, pady=10)
        self.user_search_history_box = tk.Listbox(self, height=5, width=50)
        self.user_search_history_box.grid(column=0, row=3, columnspan=2, padx=10, pady=10)

        # Trending books table
        ttk.Label(self, text="Trending Books").grid(column=0, row=4, padx=10, pady=10)
        self.trending_table = TrendingTable(self)
        self.trending_table.grid(column=0, row=5, padx=10, pady=10)

    def user_search_book(self):
        ref_number = self.user_search_input.get()
        if not ref_number.isdigit():
            messagebox.showwarning("Input Error", "Please enter a valid reference number.")
            return

        book = bst.search(int(ref_number))
        if book:
            messagebox.showinfo("Search Result", f"Book Found:\nReference Number: {book.ref_number}\nTitle: {book.title}\nAuthor: {book.author}")
            self.user_search_history_box.insert(tk.END, f"{book.ref_number} - {book.title} by {book.author}")
            self.search_history.append(book.ref_number)
        else:
            messagebox.showinfo("Search Result", "Book not found.")

class TrendingTable(ttk.Frame):
    def __init__(self, parent):
        super().__init__(parent)
        self.setup_ui()

    def setup_ui(self):
        # Trending books table
        self.trending_table = ttk.Treeview(self, columns=("Ref Number", "Title", "Author"), show='headings')
        self.trending_table.heading("Ref Number", text="Reference Number")
        self.trending_table.heading("Title", text="Title")
        self.trending_table.heading("Author", text="Author")
        self.trending_table.pack(padx=10, pady=10)

        self.load_trending_books()

    def load_trending_books(self):
        for row in self.trending_table.get_children():
            self.trending_table.delete(row)
        for book in bst.inorder_traversal():
            if book.is_trending:
                self.trending_table.insert("", "end", values=(book.ref_number, book.title, book.author))

if __name__ == "__main__":
    app = BookstoreApp()
    app.mainloop()
    save_books_to_excel()